# 🎓 Classification d'Images - Pipeline Complet
## Transfer Learning avec ResNet18 (4 Classes)

**Objectif:** Apprendre le deep learning en construisant un classifier d'images de matériel informatique

**Classes:** 🖱️ Mouse | ⌨️ Keyboard | 💻 Laptop | 🖥️ Monitor

### 🎯 Que va-t-on faire?
1. **Préparer les données** → Upload, explore, augmente
2. **Construire le modèle** → ResNet18 pré-entraîné + adaptation 4 classes
3. **Entraîner** → Transfer Learning (gèle backbone, entraîne tête)
4. **Évaluer** → Accuracy, precision, recall, F1-score
5. **Prédire** → Teste sur de nouvelles images

### 💡 Concept clé: Transfer Learning
- **ResNet18** a déjà appris à reconnaître formes, textures sur ImageNet (1M+ images)
- On **gèle** ses poids internes (pour garder ces features génériques)
- On **réentraîne** uniquement la dernière couche pour nos 4 classes spécifiques
- **Résultat:** Apprentissage rapide avec peu de données!

---

## 1️⃣ Installation des Dépendances

On va installer les bibliothèques nécessaires pour:
- **PyTorch** - Framework deep learning
- **torchvision** - Modèles pré-entraînés et outils images
- **scikit-learn** - Métriques d'évaluation
- **matplotlib/seaborn** - Visualisations
- **tqdm** - Barres de progression

In [1]:
# Installer les dépendances (déjà présentes sur Colab, mais s'assurer de la dernière version)
import subprocess
import sys

print("📦 Vérification des dépendances...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "scikit-learn", "matplotlib", "seaborn", "pillow", "tqdm"])
print("✅ Toutes les dépendances sont installées!")

📦 Vérification des dépendances...
✅ Toutes les dépendances sont installées!


In [2]:
# Imports principaux
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.datasets import ImageFolder

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import defaultdict
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from pathlib import Path

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Tous les imports sont réussis!")

✅ Tous les imports sont réussis!


In [3]:
# Déterminer le device (GPU si disponible, sinon CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Device utilisé: {device}")

if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Mémoire VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  GPU non disponible. Le CPU fonctionnera mais plus lentement (entraînement ~5-10 min).")

# Vérifier les versions
print(f"\n📚 Versions:")
print(f"  - PyTorch: {torch.__version__}")
print(f"  - NumPy: {np.__version__}")

🔧 Device utilisé: cpu
⚠️  GPU non disponible. Le CPU fonctionnera mais plus lentement (entraînement ~5-10 min).

📚 Versions:
  - PyTorch: 2.10.0+cpu
  - NumPy: 2.0.2


## 2️⃣ Upload des Données

### 📁 Structure attendue:
```
data/
├── train/
│   ├── keyboard/
│   ├── mouse/
│   ├── laptop/
│   └── monitor/
├── val/
│   ├── keyboard/
│   ├── mouse/
│   ├── laptop/
│   └── monitor/
└── test/
    ├── keyboard/
    ├── mouse/
    ├── laptop/
    └── monitor/
```

### 📝 Instructions:
1. Cliquez sur le bouton "Upload" ci-dessous
2. Sélectionnez vos images (formats: jpg, png, webp, bmp, avif)
3. **Important:** Organisez-les dans la structure ci-dessus AVANT de charger

**Conseil:** Pour débuter, vous pouvez avoir:
- 10-50 images par classe au minimum
- **Répartition recommandée:** 70% train / 15% val / 15% test par classe

In [4]:
# Créer la structure de dossiers pour les données
data_dir = "data"
splits = ["train", "val", "test"]
classes = ["keyboard", "mouse", "laptop", "monitor"]

# Créer tous les dossiers
for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(data_dir, split, cls), exist_ok=True)

print(f"✅ Structure de dossiers créée dans: {data_dir}/")
print(f"\n📂 Dossiers créés:")
for split in splits:
    for cls in classes:
        path = os.path.join(data_dir, split, cls)
        print(f"   {path}/")

✅ Structure de dossiers créée dans: data/

📂 Dossiers créés:
   data/train/keyboard/
   data/train/mouse/
   data/train/laptop/
   data/train/monitor/
   data/val/keyboard/
   data/val/mouse/
   data/val/laptop/
   data/val/monitor/
   data/test/keyboard/
   data/test/mouse/
   data/test/laptop/
   data/test/monitor/


In [5]:
# Uploader des fichiers
from google.colab import files

print("📤 Cliquez sur 'Choisir les fichiers' et sélectionnez vos images")
print("\n⚠️  Astuce: Comprimez vos images en ZIP avant d'uploader (plus rapide)")
print("Après upload, utilisez le code ci-dessous pour extraire et organiser.\n")

uploaded = files.upload()
print(f"\n✅ {len(uploaded)} fichier(s) uploadé(s)")

📤 Cliquez sur 'Choisir les fichiers' et sélectionnez vos images

⚠️  Astuce: Comprimez vos images en ZIP avant d'uploader (plus rapide)
Après upload, utilisez le code ci-dessous pour extraire et organiser.



KeyboardInterrupt: 

In [ ]:
# Si vous avez uploadé un ZIP, l'extraire
import zipfile
import shutil

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"📦 Extraction de {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('temp_extract')
        
        # Déplacer les fichiers extraits vers data/
        for root, dirs, files_list in os.walk('temp_extract'):
            for file in files_list:
                if file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.avif')):
                    src = os.path.join(root, file)
                    # Garder la structure de dossiers existante
                    rel_path = os.path.relpath(src, 'temp_extract')
                    dst = os.path.join(data_dir, rel_path)
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(src, dst)
        
        # Nettoyer
        shutil.rmtree('temp_extract')
        os.remove(filename)
        print(f"✅ Extraction complète!")

print("\n📂 Fichiers disponibles:")
for root, dirs, files_list in os.walk(data_dir):
    level = root.replace(data_dir, '').count(os.sep)
    if files_list:
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/: {len(files_list)} fichier(s)")

## 3️⃣ Exploration des Données

Analysons les images uploadées: nombre, résolutions, distribution par classe.

In [ ]:
# Compter les images par classe et split
def count_images(data_dir):
    """Compter les images dans chaque dossier"""
    stats = {}
    total = 0
    
    for split in os.listdir(data_dir):
        split_path = os.path.join(data_dir, split)
        if not os.path.isdir(split_path):
            continue
        
        stats[split] = {}
        split_total = 0
        
        for cls in os.listdir(split_path):
            cls_path = os.path.join(split_path, cls)
            if not os.path.isdir(cls_path):
                continue
            
            # Compter fichiers images
            images = [f for f in os.listdir(cls_path) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.avif'))]
            count = len(images)
            stats[split][cls] = count
            split_total += count
        
        stats[split]['TOTAL'] = split_total
        total += split_total
    
    stats['GRAND_TOTAL'] = total
    return stats

# Calculer les stats
stats = count_images(data_dir)

# Afficher les statistiques
print("📊 Statistiques du Dataset")
print("="*50)

for split in ['train', 'val', 'test']:
    if split in stats:
        print(f"\n{split.upper()}:")
        total_split = 0
        for cls, count in stats[split].items():
            if cls != 'TOTAL':
                print(f"  {cls:12s}: {count:3d} images")
                total_split += count
        print(f"  {'TOTAL':12s}: {total_split:3d} images")

print(f"\n{'TOTAL DATASET':12s}: {stats.get('GRAND_TOTAL', 0)} images")

# Vérifier le minimum recommandé
min_per_class = 10
if stats.get('GRAND_TOTAL', 0) > 0:
    avg_per_class = stats.get('GRAND_TOTAL', 0) / 4
    if avg_per_class >= min_per_class:
        print(f"✅ Dataset suffisant pour commencer (moyenne {avg_per_class:.1f}/classe)")
    else:
        print(f"⚠️  Dataset petit ({avg_per_class:.1f}/classe), mais on peut tenter!")
else:
    print("⚠️  Aucune image détectée. Vérifiez l'upload et la structure de dossiers.")

In [ ]:
# Visualiser quelques images d'exemple
def plot_sample_images(data_dir, num_per_class=2):
    """Afficher quelques images de chaque classe"""
    classes = ['keyboard', 'mouse', 'laptop', 'monitor']
    
    fig, axes = plt.subplots(4, num_per_class, figsize=(12, 10))
    fig.suptitle('📸 Exemples d\'Images par Classe', fontsize=16, fontweight='bold')
    
    for row, cls in enumerate(classes):
        # Chercher les images de cette classe
        images_paths = []
        for split in ['train', 'val', 'test']:
            cls_path = os.path.join(data_dir, split, cls)
            if os.path.exists(cls_path):
                for img_file in os.listdir(cls_path):
                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.avif')):
                        images_paths.append(os.path.join(cls_path, img_file))
                        if len(images_paths) >= num_per_class:
                            break
            if len(images_paths) >= num_per_class:
                break
        
        # Afficher les images
        for col in range(num_per_class):
            ax = axes[row, col]
            if col < len(images_paths):
                try:
                    img = Image.open(images_paths[col])
                    ax.imshow(img)
                    ax.set_title(f"{cls}\n{img.size}", fontsize=10)
                except Exception as e:
                    ax.text(0.5, 0.5, f"Erreur: {e}", ha='center', va='center')
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Afficher les samples
if stats.get('GRAND_TOTAL', 0) > 0:
    plot_sample_images(data_dir, num_per_class=2)
else:
    print("⚠️  Pas d'images à visualiser. Veuillez d'abord uploader vos données.")

In [ ]:
# Analyser les résolutions et formats
def analyze_image_properties(data_dir):
    """Analyser les propriétés des images (résolution, format, etc.)"""
    resolutions = defaultdict(int)
    formats = defaultdict(int)
    
    for root, dirs, files_list in os.walk(data_dir):
        for file in files_list:
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.avif')):
                filepath = os.path.join(root, file)
                try:
                    img = Image.open(filepath)
                    size = img.size  # (width, height)
                    res_key = f"{size[0]}×{size[1]}"
                    resolutions[res_key] += 1
                    formats[img.format or 'Unknown'] += 1
                except:
                    pass
    
    return resolutions, formats

# Analyser
resolutions, formats = analyze_image_properties(data_dir)

print("\n🖼️  Formats d'Images:")
for fmt, count in sorted(formats.items(), key=lambda x: -x[1]):
    print(f"  {fmt:8s}: {count:3d} images")

print("\n📐 Résolutions (Top 10):")
for res, count in sorted(resolutions.items(), key=lambda x: -x[1])[:10]:
    print(f"  {res:15s}: {count:3d} images")

if resolutions:
    avg_res = list(resolutions.keys())[0]
    print(f"\n💡 Astuce: Les images seront redimensionnées à 224×224 (standard ResNet18)")

## 4️⃣ Préparation des Données (Data Pipeline)

### 🔄 Étapes de préparation:
1. **Data Augmentation** - Augmenter la diversité des images (crop, flip, rotation, couleur)
2. **Normalization** - Normaliser avec les stats ImageNet (moyenne/std)
3. **DataLoaders** - Créer des batches pour l'entraînement

### 💡 Pourquoi l'augmentation?
- Augmente artificiellement la taille du dataset
- Rend le modèle plus robuste (rotation, zoom, couleurs différentes)
- Réduit l'overfitting

In [ ]:
# Définir les transformations (augmentation + normalization)
# Valeurs de normalization ImageNet (pré-calculées sur 1M+ images)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224  # Taille standard ResNet18

# Transformations pour l'ENTRAÎNEMENT (avec augmentation)
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),  # Crop aléatoire + resize
    transforms.RandomHorizontalFlip(p=0.5),                       # Flip horizontal 50% du temps
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # Variation couleur
    transforms.RandomRotation(10),                                # Rotation légère
    transforms.ToTensor(),                                        # Convertir en tensor PyTorch
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)    # Normaliser
])

# Transformations pour VALIDATION et TEST (pas d'augmentation, juste resize + normalize)
val_test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),                  # Resize simple
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("✅ Transformations définies:")
print(f"\n📐 Taille images: {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"📊 Normalisation: ImageNet (mean={IMAGENET_MEAN}, std={IMAGENET_STD})")

In [ ]:
# Créer les datasets
try:
    train_dataset = ImageFolder(os.path.join(data_dir, 'train'), transform=train_transforms)
    val_dataset = ImageFolder(os.path.join(data_dir, 'val'), transform=val_test_transforms)
    test_dataset = ImageFolder(os.path.join(data_dir, 'test'), transform=val_test_transforms)
    
    # Créer les DataLoaders
    batch_size = 16  # Nombre d'images par batch
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Récupérer le mapping classe -> idx
    class_to_idx = train_dataset.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    
    print("✅ Datasets créés avec succès!")
    print(f"\n📊 Tailles:")
    print(f"  Train: {len(train_dataset)} images")
    print(f"  Val:   {len(val_dataset)} images")
    print(f"  Test:  {len(test_dataset)} images")
    
    print(f"\n🏷️  Classes:")
    for cls, idx in sorted(class_to_idx.items()):
        print(f"  {idx}: {cls}")
    
    print(f"\n📦 Batch size: {batch_size}")
    print(f"  Train: {len(train_loader)} batches")
    print(f"  Val:   {len(val_loader)} batches")
    print(f"  Test:  {len(test_loader)} batches")
    
except Exception as e:
    print(f"❌ Erreur lors de la création des datasets: {e}")
    print("Vérifiez que les dossiers data/train, data/val, data/test existent avec des sous-dossiers par classe.")

In [ ]:
# Visualiser un batch d'images (avant et après augmentation)
def visualize_batch(train_loader, val_loader, idx_to_class):
    """Afficher un batch d'images (train avec augmentation vs val sans augmentation)"""
    
    # Denormalization function pour afficher les images correctement
    def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
        for t, m, s in zip(tensor, mean, std):
            t.mul_(s).add_(m)
        return tensor.clamp(0, 1)
    
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    fig.suptitle('📸 Batch d\'Images (Train avec Augmentation vs Val)', fontsize=14, fontweight='bold')
    
    # Train batch (avec augmentation)
    train_batch, train_labels = next(iter(train_loader))
    for i in range(min(4, len(train_batch))):
        img = denormalize(train_batch[i].cpu().clone())
        img = img.permute(1, 2, 0).numpy()
        axes[0, i].imshow(img)
        axes[0, i].set_title(f"Train: {idx_to_class[train_labels[i].item()]}")
        axes[0, i].axis('off')
    
    # Val batch (sans augmentation)
    val_batch, val_labels = next(iter(val_loader))
    for i in range(min(4, len(val_batch))):
        img = denormalize(val_batch[i].cpu().clone())
        img = img.permute(1, 2, 0).numpy()
        axes[1, i].imshow(img)
        axes[1, i].set_title(f"Val: {idx_to_class[val_labels[i].item()]}")
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Afficher le batch
if len(train_loader) > 0:
    visualize_batch(train_loader, val_loader, idx_to_class)
else:
    print("⚠️  Pas assez d'images pour visualiser un batch.")

## 5️⃣ Construction du Modèle

### 🧠 Architecture ResNet18 + Transfer Learning

**ResNet18 standard:**
```
Input (3 channels) 
  ↓
Conv1 + BatchNorm
  ↓
Layer1 (2×Basic Block) ← GELÉES (Transfer Learning)
  ↓
Layer2 (2×Basic Block) ← GELÉES
  ↓
Layer3 (2×Basic Block) ← GELÉES
  ↓
Layer4 (2×Basic Block) ← GELÉES
  ↓
Global Average Pooling (512 features)
  ↓
FC Layer: [512 → 4 classes] ← ENTRAÎNABLE ✏️
```

**Idée:** Geler le backbone (déjà expert sur ImageNet) et réentraîner la tête pour nos 4 classes!

In [ ]:
# Construire le modèle ResNet18 + adaptation 4 classes
def build_model(num_classes=4, pretrained=True, fine_tune=False):
    """
    Construire ResNet18 pour classification multi-classe
    
    Args:
        num_classes: Nombre de classes (4)
        pretrained: Charger poids ImageNet (True) ou initialiser aléatoire (False)
        fine_tune: Si True, dégeler layer4 en plus de la FC
    
    Returns:
        model: ResNet18 adapté
    """
    # Charger ResNet18 pré-entraîné
    model = models.resnet18(pretrained=pretrained)
    
    # TRANSFER LEARNING: Geler tous les paramètres
    for param in model.parameters():
        param.requires_grad = False
    
    # FINE-TUNING (optionnel): Dégeler layer4
    if fine_tune:
        for param in model.layer4.parameters():
            param.requires_grad = True
    
    # Remplacer la couche FC finale pour nos 4 classes
    # ResNet18 a 512 features en sortie de layer4
    num_features = model.fc.in_features  # 512
    model.fc = nn.Linear(num_features, num_classes)
    
    # ⚠️ IMPORTANT: Les paramètres de la nouvelle FC ne sont pas gelés!
    # Elle s'entraînera automatiquement
    
    return model

# Construire le modèle
model = build_model(num_classes=len(class_to_idx), pretrained=True, fine_tune=False)
model = model.to(device)

print("✅ Modèle ResNet18 créé avec succès!")
print(f"\n📊 Architecture:")
print(f"  Backbone: Gelé (Transfer Learning)")
print(f"  FC finale: [512 → {len(class_to_idx)}] (entraînable)")

In [ ]:
# Analyser les paramètres du modèle
def count_parameters(model):
    """Compter paramètres totaux et entraînables"""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    return total, trainable, frozen

total, trainable, frozen = count_parameters(model)

print("\n📈 Paramètres du Modèle:")
print(f"  Total:      {total:,} paramètres")
print(f"  Entraînable: {trainable:,} ({100*trainable/total:.1f}%)")
print(f"  Gelé:       {frozen:,} ({100*frozen/total:.1f}%)")

print(f"\n💡 Avantage Transfer Learning:")
print(f"  - Seulement {100*trainable/total:.1f}% des paramètres à entraîner")
print(f"  - Convergence rapide et stable")
print(f"  - Pas besoin de beaucoup de données")

In [ ]:
# Configurer l'optimiseur et la loss function
criterion = nn.CrossEntropyLoss()  # Loss pour classification multi-classe
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),  # Optimiser seulement les param entraînables
    lr=0.001,  # Learning rate
    weight_decay=1e-4  # Régularisation L2 (évite overfitting)
)

print("✅ Optimiseur et Loss configurés:")
print(f"  Loss: CrossEntropyLoss (standard pour classification)")
print(f"  Optimizer: Adam")
print(f"  Learning Rate: 0.001")
print(f"  Weight Decay: 1e-4 (L2 regularization)")

## 6️⃣ Entraînement du Modèle

### 🔄 Boucle d'entraînement standard:
```
Pour chaque epoch:
  1. Itérer sur train_loader (par batches)
  2. Forward pass: y_pred = model(x)
  3. Calculer loss: L = criterion(y_pred, y_true)
  4. Backward pass: loss.backward()
  5. Update: optimizer.step()
  
  6. Évaluer sur val_loader
  7. Sauvegarder si val_acc > best_acc
```

### 📊 Métriques à surveiller:
- **Loss** (diminue = bon apprentissage)
- **Accuracy** (augmente = meilleure classification)
- **Val Accuracy** (doit suivre train, sinon overfitting)

In [ ]:
# Fonctions d'entraînement et évaluation
def train_epoch(model, train_loader, criterion, optimizer, device):
    """
    Une epoch d'entraînement
    
    Returns:
        avg_loss: Loss moyenne de l'epoch
        accuracy: Accuracy de l'epoch
    """
    model.train()  # Mode entraînement
    total_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc="Training", leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()  # Réinitialiser gradients
        loss.backward()        # Calculer gradients
        optimizer.step()       # Mettre à jour poids
        
        # Statistiques
        total_loss += loss.item() * labels.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
        
        pbar.set_postfix({'loss': f'{total_loss/total:.4f}', 'acc': f'{100*correct/total:.1f}%'})
    
    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, val_loader, criterion, device):
    """
    Évaluer le modèle sur validation/test set
    
    Returns:
        avg_loss: Loss moyenne
        accuracy: Accuracy
        all_preds: Toutes les prédictions (pour confusion matrix)
        all_labels: Tous les labels vrais
    """
    model.eval()  # Mode évaluation
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():  # Pas besoin de calculer gradients en évaluation
        pbar = tqdm(val_loader, desc="Evaluating", leave=False)
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * labels.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': f'{total_loss/total:.4f}', 'acc': f'{100*correct/total:.1f}%'})
    
    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy, np.array(all_preds), np.array(all_labels)

print("✅ Fonctions d'entraînement définies!")

In [ ]:
# Boucle d'entraînement principale
num_epochs = 8  # Nombre d'epochs
best_val_acc = 0.0
best_model_path = 'best_model.pth'

# Pour tracer les courbes
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

print(f"🚀 Commençant l'entraînement pour {num_epochs} epochs...\n")

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print("="*50)
    
    # Entraînement
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    
    # Validation
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {100*train_acc:.1f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {100*val_acc:.1f}%")
    
    # Sauvegarder le meilleur modèle
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"✅ Modèle sauvegardé! (Val Acc: {100*val_acc:.1f}%)")
    else:
        print(f"(Meilleure Val Acc: {100*best_val_acc:.1f}%)")

print(f"\n🎉 Entraînement terminé!")
print(f"Meilleure Val Accuracy: {100*best_val_acc:.1f}%")
print(f"Modèle sauvegardé: {best_model_path}")

In [ ]:
# Tracer les courbes d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('📉 Loss au fil des Epochs')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot([100*x for x in history['train_acc']], label='Train Accuracy', marker='o')
axes[1].plot([100*x for x in history['val_acc']], label='Val Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('📈 Accuracy au fil des Epochs')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Graphiques sauvegardés: training_curves.png")

In [ ]:
# Sauvegarder le mapping class_to_idx
with open('class_to_idx.json', 'w') as f:
    json.dump(class_to_idx, f, indent=2)

print("✅ Mapping classes sauvegardé: class_to_idx.json")

## 7️⃣ Évaluation sur le Test Set

Évaluer la performance finale du modèle sur les données de test (jamais vues pendant l'entraînement)

In [ ]:
# Charger le meilleur modèle
model.load_state_dict(torch.load(best_model_path))
model.eval()

print("✅ Meilleur modèle chargé")

# Évaluer sur test set
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

print(f"\n📊 Performance sur Test Set:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {100*test_acc:.1f}%")

In [ ]:
# Classification Report détaillé
report = classification_report(
    test_labels, 
    test_preds, 
    target_names=[idx_to_class[i] for i in range(len(class_to_idx))],
    digits=4
)

print("\n📈 Classification Report:")
print("="*60)
print(report)

# Sauvegarder
with open('classification_report.txt', 'w') as f:
    f.write(report)
print("✅ Rapport sauvegardé: classification_report.txt")

In [ ]:
# Matrice de confusion
cm = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=[idx_to_class[i] for i in range(len(class_to_idx))],
    yticklabels=[idx_to_class[i] for i in range(len(class_to_idx))],
    ax=ax,
    cbar_kws={'label': 'Nombre d\'images'}
)
ax.set_xlabel('Prédiction')
ax.set_ylabel('Vrai Label')
ax.set_title('🔥 Matrice de Confusion - Test Set')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Matrice de confusion sauvegardée: confusion_matrix.png")

## 8️⃣ Prédictions sur Nouvelles Images

Charger une image et faire une prédiction avec le modèle entraîné

In [ ]:
# Fonction de prédiction sur une image
def predict_image(image_path, model, device, idx_to_class):
    """
    Prédire la classe d'une image
    
    Returns:
        predicted_class: Classe prédite
        confidence: Confiance (en %)
        all_confidences: Dict avec confiances pour toutes classes
    """
    # Charger et traiter l'image
    image = Image.open(image_path).convert('RGB')
    image_tensor = val_test_transforms(image).unsqueeze(0).to(device)
    
    # Prédiction
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.softmax(output, dim=1)[0].cpu().numpy()
        predicted_idx = np.argmax(probabilities)
    
    predicted_class = idx_to_class[predicted_idx]
    confidence = probabilities[predicted_idx] * 100
    
    # Toutes les confiances (top-3)
    top_3_idx = np.argsort(probabilities)[::-1][:3]
    all_confidences = {
        idx_to_class[i]: probabilities[i] * 100 
        for i in top_3_idx
    }
    
    return image, predicted_class, confidence, all_confidences

print("✅ Fonction de prédiction définie!")

In [ ]:
# Tester sur un échantillon du test set
if len(test_dataset) > 0:
    # Prendre une image aléatoire du test set
    test_image_idx = np.random.randint(0, len(test_dataset))
    test_image_path = test_dataset.imgs[test_image_idx][0]
    true_class = idx_to_class[test_dataset.imgs[test_image_idx][1]]
    
    # Prédire
    image, pred_class, confidence, all_conf = predict_image(
        test_image_path, model, device, idx_to_class
    )
    
    # Afficher
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Image
    axes[0].imshow(image)
    axes[0].set_title(f"Image de Test\n(Vrai: {true_class})")
    axes[0].axis('off')
    
    # Prédictions (top-3)
    classes_list = list(all_conf.keys())
    confidences_list = list(all_conf.values())
    colors = ['green' if c == pred_class else 'gray' for c in classes_list]
    
    axes[1].barh(classes_list, confidences_list, color=colors)
    axes[1].set_xlabel('Confiance (%)')
    axes[1].set_title(f"Top-3 Prédictions\n✅ Prédiction: {pred_class} ({confidence:.1f}%)")
    axes[1].set_xlim([0, 100])
    
    # Ajouter les pourcentages
    for i, (cls, conf) in enumerate(all_conf.items()):
        axes[1].text(conf + 2, i, f'{conf:.1f}%', va='center')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Résultat:")
    print(f"  Vrai classe: {true_class}")
    print(f"  Prédiction: {pred_class} ✅" if pred_class == true_class else f"  Prédiction: {pred_class} ❌")
    print(f"  Confiance: {confidence:.1f}%")
    print(f"\nTop-3:")
    for cls, conf in all_conf.items():
        print(f"  {cls:12s}: {conf:5.1f}%")
else:
    print("⚠️  Pas de test images disponibles")

In [ ]:
# Upload une nouvelle image et la prédire
from google.colab import files

print("📤 Uploadez une nouvelle image pour faire une prédiction")
print("Formats supportés: jpg, png, webp, bmp, avif\n")

uploaded = files.upload()

if len(uploaded) > 0:
    for filename in uploaded.keys():
        if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.avif')):
            print(f"\n📸 Prédiction pour: {filename}")
print("="*50)

image, pred_class, confidence, all_conf = predict_image(
filename, model, device, idx_to_class
)

# Afficher
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Image
axes[0].imshow(image)
axes[0].set_title(f"Votre Image")
axes[0].axis('off')

# Prédictions
classes_list = list(all_conf.keys())
confidences_list = list(all_conf.values())
colors = ['green' if c == pred_class else 'gray' for c in classes_list]

axes[1].barh(classes_list, confidences_list, color=colors)
axes[1].set_xlabel('Confiance (%)')
axes[1].set_title(f"Prédictions\n✅ Classe: {pred_class} ({confidence:.1f}%)")
axes[1].set_xlim([0, 100])

for i, (cls, conf) in enumerate(all_conf.items()):
    axes[1].text(conf + 2, i, f'{conf:.1f}%', va='center')

plt.tight_layout()
plt.show()

print(f"\n🎯 Résultat:")
print(f"  Classe prédite: {pred_class}")
print(f"  Confiance: {confidence:.1f}%")
print(f"\n📊 Toutes les confiances:")
for cls, conf in all_conf.items():
    print(f"  {cls:12s}: {conf:5.1f}%")
else:
    print("⚠️  Aucun fichier uploadé")

## 9️⃣ (BONUS) Fine-Tuning - Améliorer le Modèle

### 🎯 Concept:
Si vous avez des résultats insuffisants, vous pouvez "dégeler" quelques couches du backbone pour les réentraîner.

**Avant (Transfer Learning):**
```
Backbone ResNet18 [GELÉ]
FC Layer [ENTRAÎNABLE]
```

**Après (Fine-Tuning):**
```
Layers 1-3 [GELÉS]
Layer 4 [ENTRAÎNABLE]  ← Dégelé
FC Layer [ENTRAÎNABLE]
```

### ⚠️ Attention:
- Fine-tuning nécessite plus de temps
- Learning rate doit être plus petit (on utilise déjà de bons poids)
- Besoin de plus de données pour ne pas overfitter

In [ ]:
# (OPTIONNEL) Fine-tuning
PERFORM_FINE_TUNE = False  # Mettez à True pour activer

if PERFORM_FINE_TUNE and len(train_dataset) > 100:
    print("🔧 Démarrage Fine-Tuning (dégeler layer4)...\n")
    
    # Dégeler layer4
    for param in model.layer4.parameters():
        param.requires_grad = True
    
    # Optimiseur avec learning rate plus petit
    optimizer_ft = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.0001,  # 10x plus petit
        weight_decay=1e-4
    )
    
    # Entraîner 4 epochs supplémentaires
    num_epochs_ft = 4
    best_val_acc_ft = best_val_acc
    
    for epoch in range(num_epochs_ft):
        print(f"Fine-Tune Epoch [{epoch+1}/{num_epochs_ft}]")

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer_ft, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {100*train_acc:.1f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {100*val_acc:.1f}%")

if val_acc > best_val_acc_ft:
    best_val_acc_ft = val_acc
    torch.save(model.state_dict(), best_model_path)
    print(f"✅ Meilleur modèle mis à jour!")
    print()

    print("✅ Fine-tuning terminé!")
else:
    print("ℹ️  Fine-tuning désactivé (recommandé uniquement avec beaucoup de données)")

## 🎉 Conclusion

### 📊 Résumé du Pipeline

Félicitations! Vous avez complété un pipeline complet de Deep Learning:

1. ✅ **Données** - Upload, exploration, augmentation
2. ✅ **Modèle** - ResNet18 + Transfer Learning
3. ✅ **Entraînement** - 8 epochs avec validation
4. ✅ **Évaluation** - Métriques, confusion matrix
5. ✅ **Prédiction** - Classification de nouvelles images
6. ✅ **(Bonus)** - Fine-tuning possible

### 📁 Fichiers Générés
- `best_model.pth` - Poids du modèle entraîné
- `class_to_idx.json` - Mapping classe → index
- `training_curves.png` - Graphiques loss/accuracy
- `confusion_matrix.png` - Matrice de confusion
- `classification_report.txt` - Métriques détaillées

### 💡 Prochaines Améliorations
- Collecter plus d'images pour chaque classe
- Tester d'autres architectures (EfficientNet, Vision Transformer)
- Ajuster hyperparamètres (learning rate, batch size)
- Ajouter plus d'augmentation pour réduire overfitting
- Exporter en ONNX pour déploiement edge

### 📚 Concepts Clés Appris
- **Transfer Learning**: Réutiliser modèles pré-entraînés
- **Data Augmentation**: Augmenter diversité du dataset
- **Batch Processing**: Traiter par batches pour efficacité
- **Train/Val/Test Splits**: Évaluation rigoureuse
- **Métriques**: Accuracy, precision, recall, F1-score

---

**Bon apprentissage! 🚀🎓**